# 🚀 LLM Inference Engine — Quickstart
**Model**: Microsoft Phi-3 Mini 3.8B  
**Hardware**: Kaggle T4 GPU (16GB VRAM)  
**Goal**: Load model, run inference, verify setup

> Enable GPU in Kaggle: Settings → Accelerator → GPU T4 x2

In [ ]:
# Step 1 — Install dependencies
!pip install -q transformers accelerate bitsandbytes optimum fastapi uvicorn mlflow pyyaml seaborn

In [ ]:
# Step 2 — Clone the repo
import os
if not os.path.exists('llm-inference-engine'):
    !git clone https://github.com/yourusername/llm-inference-engine
%cd llm-inference-engine
!pip install -e . -q

In [ ]:
# Step 3 — Detect hardware
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'PyTorch CUDA version: {torch.version.cuda}')

In [ ]:
# Step 4 — Load Phi-3 Mini in FP16
from src.engine import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(
    model_name='microsoft/Phi-3-mini-4k-instruct',
    precision='fp16',
    cache_dir='/kaggle/working/models',
)
print('✅ Model loaded successfully!')

In [ ]:
# Step 5 — Check VRAM usage after loading
from src.utils import get_gpu_memory_usage
mem = get_gpu_memory_usage()
print(f"VRAM allocated: {mem['allocated_gb']:.2f} GB")
print(f"VRAM free:      {mem['free_gb']:.2f} GB")
print(f"VRAM total:     {mem['total_gb']:.2f} GB")

In [ ]:
# Step 6 — Run inference
from src.engine import generate

prompt = """<|user|>
Explain the key difference between CUDA and ROCm for GPU computing in 3 bullet points.
<|end|>
<|assistant|>"""

result = generate(
    model, tokenizer, prompt,
    max_new_tokens=256,
    temperature=0.7,
)

print('=== OUTPUT ===')
print(result['text'])
print(f"\n📊 Tokens generated : {result['tokens_generated']}")
print(f"⚡ Latency          : {result['latency_ms']:.0f} ms")
print(f"🚀 Tokens/second    : {result['tokens_per_second']:.1f}")

In [ ]:
# Step 7 — Test a few different prompts
test_prompts = [
    'Write a Python function to compute cosine similarity between two vectors.',
    'What is the intuition behind attention in transformers?',
    'Explain INT8 quantization for neural networks in simple terms.',
]

for p in test_prompts:
    r = generate(model, tokenizer, p, max_new_tokens=150)
    print(f'Prompt: {p[:60]}...')
    print(f'  → {r["tokens_per_second"]:.1f} tok/s | {r["latency_ms"]:.0f}ms')
    print()

## ✅ Setup complete!
Proceed to **Notebook 02** for full benchmarking, or **Notebook 03** for quantization comparison.
